# EDA Report

This notebook performs exploratory data analysis (EDA) for **Tasks 1–4**.

**How to use**
- First download data into `data/` (see the download cell below).
- Run cells top-to-bottom.
- Record your key findings in the final “Summary / Decisions” section and then copy the relevant conclusions into each `Task-k-report.ipynb`.

**Data files expected (local paths)**
- `data/train.h5`
- `data/events.csv`
- `data/surprise-task{k}.h5` and `data/surprise-events{k}.csv`


In [ ]:
import sys; sys.path.insert(0, "..")

In [ ]:
# Setup
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from predicting_unpredictable import data, preprocess, split, submission

SEED = 20260126
rng = np.random.default_rng(SEED)

DATA_DIR = Path("data")
TRAIN_H5 = DATA_DIR / "train.h5"
EVENTS_CSV = DATA_DIR / "events.csv"

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 120

print("DATA_DIR:", DATA_DIR.resolve())
print("train.h5 exists:", TRAIN_H5.exists())
print("events.csv exists:", EVENTS_CSV.exists())


In [ ]:
# Download data from HuggingFace (run in Colab or with network access)
# This cell is safe to skip if you already downloaded into DATA_DIR.
try:
    data.download_training_data(local_dir=DATA_DIR)
    data.download_surprise_data(local_dir=DATA_DIR)
except ModuleNotFoundError as e:
    print("Optional download skipped (missing dependency):", e)
    print("Tip: install deps with `pip install -r requirements.txt`.")
except Exception as e:
    print("Optional download failed:", repr(e))


## 1) Data contract checks (events.csv)

Goal: verify row counts, uniqueness, and label set. This prevents silent mistakes later.


In [ ]:
df = data.read_events_csv(EVENTS_CSV)

print("rows:", len(df))
print("unique storm ids:", df["id"].nunique())
print("img_type unique:", sorted(df["img_type"].astype(str).unique().tolist()))
print("event_type unique:", sorted(df["event_type"].astype(str).unique().tolist()))

# Expectation: 800 storms, 5 img_types per storm => 4000 rows.
counts_per_id = df.groupby("id").size()
print("rows per id (min/max):", int(counts_per_id.min()), int(counts_per_id.max()))
print("ids with !=5 rows:", int((counts_per_id != 5).sum()))

# start_utc: lght rows typically have NaT
is_lght = df["img_type"].astype(str) == "lght"
print("start_utc NaT ratio (all):", float(df["start_utc"].isna().mean()))
print("start_utc NaT ratio (lght):", float(df.loc[is_lght, "start_utc"].isna().mean()))
print("start_utc NaT ratio (non-lght):", float(df.loc[~is_lght, "start_utc"].isna().mean()))

df.head()

## 2) Data contract checks (train.h5)

We only sample a few storms to confirm shapes/dtypes.


In [ ]:
sample_ids = rng.choice(df["id"].unique(), size=3, replace=False).tolist()
print("sample storm ids:", sample_ids)

img_types = ["vis", "ir069", "ir107", "vil", "lght"]

with data.open_h5(TRAIN_H5) as f:
    for sid in sample_ids:
        arrays = data.load_event_arrays(f, storm_id=sid, img_types=img_types)
        print("\n---", sid, "---")
        for k in img_types:
            a = arrays[k]
            print(f"{k:5s} shape={a.shape} dtype={a.dtype}")


## 3) Global EDA (useful for all tasks)

### 3.0 Data coverage & pipeline-friendly checks

Goal: make sure the dataset is *usable* and that our pipeline can hard-assert the expected structure.

- **events.csv**: each storm `id` should appear exactly 5 times (one per `img_type`).
- **train.h5**: for each storm `id`, group exists and contains datasets `vis, ir069, ir107, vil, lght` with expected shapes.
- **Note**: `start_utc` being `NaT` for `lght` rows is expected and should not be treated as missing metadata.

### 3.1 Storm-level class balance


In [ ]:
# --- 3.0 Data coverage & pipeline-friendly checks ---
expected_img_types = ["ir069", "ir107", "lght", "vil", "vis"]

# events.csv coverage: should be complete 5 rows per id (one per img_type)
events_counts = (
    df.pivot_table(index="id", columns="img_type", values="event_type", aggfunc="size", fill_value=0)
    .reindex(columns=expected_img_types, fill_value=0)
)

bad_ids = events_counts[(events_counts.sum(axis=1) != 5) | (events_counts.min(axis=1) != 1)]
print("events.csv coverage: storms=", len(events_counts), "bad_ids=", len(bad_ids))
if len(bad_ids) > 0:
    display(bad_ids.head(10))

# train.h5 coverage: keys and shapes (cheap: only reads dataset metadata)
expected_shapes = {
    "vis": (384, 384, 36),
    "vil": (384, 384, 36),
    "ir069": (192, 192, 36),
    "ir107": (192, 192, 36),
}

issues = []
with data.open_h5(TRAIN_H5) as f:
    for sid in df["id"].astype(str).unique():
        if sid not in f:
            issues.append((sid, "missing_group", None))
            continue
        grp = f[sid]
        for k in expected_img_types:
            if k not in grp:
                issues.append((sid, "missing_dataset", k))
                continue
            if k in expected_shapes:
                shp = tuple(grp[k].shape)
                if shp != expected_shapes[k]:
                    issues.append((sid, "bad_shape", f"{k} {shp}"))

print("train.h5 coverage issues:", len(issues))
if issues:
    display(pd.DataFrame(issues, columns=["id", "issue", "detail"]).head(20))


# --- 3.1 Storm-level class balance ---
# Use one label per storm id
storm_labels = (
    df.groupby("id", as_index=False)
    .first()[["id", "event_type"]]
    .sort_values("event_type")
)

counts = storm_labels["event_type"].value_counts().sort_values(ascending=False)
counts.plot(kind="bar")
plt.title("Storm-level event_type counts")
plt.ylabel("# storms")
plt.tight_layout()
plt.show()

counts


#### Conclusion
##### Data Integrity Checks
- **Key Observation:** Some storm IDs have incomplete image coverage (missing datasets or shape mismatches) and inconsistent resolutions (192×192 vs. 384×384).
- **Preprocessing Takeaways:**
1. Filter out invalid IDs with missing data or mismatched shapes to ensure input validity for all tasks.
2. Unify image resolution (upsample 192×192 data to 384×384 or downsample to 192×192) and standardize sequence length (pad/crop to 36 frames).
3. Further filter data by task requirements (e.g., retain storms with complete vis/ir069/ir107 for Task 2).

##### Class Imbalance Implications (Critical for Task 3)
- **Key Observation:** Severe imbalance exists, with "Thunderstorm Wind" and "Hail" as dominant classes, while "Heavy Rain", "Funnel Cloud", and "Lightning" are rare.
- **Preprocessing & Training Takeaways:**
1. Use **resampling techniques** (oversample minority classes with time-series augmentation, or undersample majority classes) and **weighted cross-entropy loss** to mitigate bias.
2. Split data with stratified sampling to preserve class distribution in train/validation sets.
3. Evaluate with F1-score/AUC-ROC instead of accuracy, and use attention-based models to focus on minority class features.

### 3.2 Time and location metadata


In [ ]:
vil_rows = df[df["img_type"].astype(str) == "vil"].copy()

# Time distribution
vil_times = vil_rows["start_utc"].dropna()
vil_times.dt.tz_localize(None).hist(bins=60)
plt.title("start_utc distribution (vil rows)")
plt.xlabel("time")
plt.tight_layout()
plt.show()

# Location scatter (approx storm positions)
plt.figure(figsize=(6, 4))
plt.scatter(vil_rows["llcrnrlon"], vil_rows["llcrnrlat"], s=5, alpha=0.4)
plt.title("Storm lower-left corner (vil rows)")
plt.xlabel("llcrnrlon")
plt.ylabel("llcrnrlat")
plt.tight_layout()
plt.show()

# Basic metadata range checks (pipeline-friendly)
print("lat range:", float(vil_rows["llcrnrlat"].min()), "..", float(vil_rows["urcrnrlat"].max()))
print("lon range:", float(vil_rows["llcrnrlon"].min()), "..", float(vil_rows["urcrnrlon"].max()))

invalid_bbox = vil_rows[(vil_rows["urcrnrlat"] <= vil_rows["llcrnrlat"]) | (vil_rows["urcrnrlon"] <= vil_rows["llcrnrlon"])].copy()
print("invalid bbox rows:", len(invalid_bbox))
if len(invalid_bbox) > 0:
    display(invalid_bbox[["id", "llcrnrlat", "llcrnrlon", "urcrnrlat", "urcrnrlon"]].head(10))

# Width/height distribution (meters)
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
vil_rows["width_m"].hist(bins=50)
plt.title("width_m (vil rows)")
plt.subplot(1, 2, 2)
vil_rows["height_m"].hist(bins=50)
plt.title("height_m (vil rows)")
plt.tight_layout()
plt.show()


#### Conclusion
- **Temporal Imbalance:** Storm events cluster heavily in September 2018 and May–July 2019, with very few samples from December 2018–February 2019.
- **Generalization Need:** Temporal stratified sampling ensures both training/validation sets include samples from all time periods, preserving seasonal diversity and enabling the model to generalize across the full annual cycle of storms.

### 3.3 Pixel-value distributions (sampled)

We sample a few storms and a few frames to inspect distributions. Use this to decide:
- z-score vs robust scaling vs clipping
- whether to rescale `vil` by `/255.0`


In [ ]:
def sample_pixels_hwt(a: np.ndarray, *, n_pixels: int = 200_000) -> np.ndarray:
    """Sample pixel values from an (H,W,T) array."""

    h, w, t = a.shape
    # sample (y,x,t) indices
    idx = rng.integers(0, h * w * t, size=min(n_pixels, h * w * t))
    y = idx // (w * t)
    rem = idx % (w * t)
    x = rem // t
    ti = rem % t
    return a[y, x, ti]


storm_id = str(rng.choice(df["id"].unique()))
print("Example storm:", storm_id)

with data.open_h5(TRAIN_H5) as f:
    arrays = data.load_event_arrays(
        f,
        storm_id=storm_id,
        img_types=["vis", "ir069", "ir107", "vil"],
    )

samples = {
    "vis": sample_pixels_hwt(arrays["vis"].astype(np.float32)),
    "ir069": sample_pixels_hwt(arrays["ir069"].astype(np.float32)),
    "ir107": sample_pixels_hwt(arrays["ir107"].astype(np.float32)),
    "vil": sample_pixels_hwt(arrays["vil"].astype(np.float32)),
}

fig, axs = plt.subplots(1, 4, figsize=(14, 3))
for ax, (k, v) in zip(axs, samples.items()):
    ax.hist(v, bins=80)
    ax.set_title(k)
plt.tight_layout()
plt.show()

for k, v in samples.items():
    v = v.astype(np.float64)
    print(k, "min/mean/max:", float(v.min()), float(v.mean()), float(v.max()))

Conclusion
- **Modality-Specific Normalization**
Normalize vis to [0,1], ir069/ir107 to [-1,1], and vil (log-transformed first) to [0,1] to preserve key meteorological features and avoid feature suppression.
- **Weighted Loss for Vil**
Use weighted L1/Focal Loss for vil’s right-skewed distribution to prioritize predicting high-value (heavy precipitation) pixels and correct prediction bias.

## 4) Task-specific EDA

### Task 1: VIL 12→12

**EDA focus → preprocessing decision**
- **Temporal distribution drift**: per-frame quantiles/mean over `t=0..35` to spot saturation or scale drift.
- **Frame-to-frame change**: stats of `|vil[t+1]-vil[t]|` to gauge storm evolution speed.
- **Spatial intensity**: thresholded area fraction (e.g. `VIL>20/40/80`) to detect “empty frames” vs strong cores.

**Slicing & normalization**
- **Input/target**: `x = t=0..11`, `y = t=12..23`.
- **Normalization**: `vil_uint8 -> float32`, `vil = vil/255.0` (default). If long-tail issues appear, consider quantile clipping, but keep the default simple.
- **Loss**: prefer `L1/Huber` for robustness to extremes.

- Visualise first 12 frames vs next 12 frames
- Quick baseline: persistence (repeat last input frame)


In [ ]:
sid = str(rng.choice(df["id"].unique()))
print("Task1 example storm:", sid)

with data.open_h5(TRAIN_H5) as f:
    vil = f[sid]["vil"][:]  # (384,384,36) uint8

x_in = vil[:, :, :12].astype(np.float32) / 255.0
x_tgt = vil[:, :, 12:24].astype(np.float32) / 255.0

# Visualise a few frames
frames = [0, 5, 11]
fig, axs = plt.subplots(2, len(frames), figsize=(12, 5))
for j, ti in enumerate(frames):
    axs[0, j].imshow(x_in[:, :, ti], vmin=0, vmax=1, cmap="turbo")
    axs[0, j].set_title(f"input t={ti+1}")
    axs[0, j].axis("off")
    axs[1, j].imshow(x_tgt[:, :, ti], vmin=0, vmax=1, cmap="turbo")
    axs[1, j].set_title(f"target t={ti+13}")
    axs[1, j].axis("off")
plt.tight_layout()
plt.show()

# Persistence baseline: repeat last input frame
pred_persist = np.repeat(x_in[:, :, 11:12], repeats=12, axis=2)
mae = float(np.mean(np.abs(pred_persist - x_tgt)))
print("Persistence baseline MAE (in [0,1] space):", mae)

# --- Extra EDA: temporal drift, frame-diff, spatial intensity ---
vil_f = vil.astype(np.float32)
T = vil_f.shape[2]

# Per-frame moments + quantiles
qs = [0.01, 0.5, 0.9, 0.99]
frame_mean = vil_f.reshape(-1, T).mean(axis=0)
frame_std = vil_f.reshape(-1, T).std(axis=0)
frame_q = np.stack([np.quantile(vil_f[:, :, t], qs) for t in range(T)], axis=0)

plt.figure(figsize=(10, 3))
plt.plot(frame_mean, label="mean")
plt.fill_between(np.arange(T), frame_mean - frame_std, frame_mean + frame_std, alpha=0.2, label="±1 std")
plt.title("Task1: VIL per-frame mean/std (uint8)")
plt.xlabel("frame t")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3))
for qi, qv in enumerate(qs):
    plt.plot(frame_q[:, qi], label=f"p{int(qv*100)}")
plt.title("Task1: VIL per-frame quantiles (uint8)")
plt.xlabel("frame t")
plt.legend()
plt.tight_layout()
plt.show()

# Frame-to-frame change intensity
abs_diff = np.abs(vil_f[:, :, 1:] - vil_f[:, :, :-1])
diff_mean = abs_diff.reshape(-1, T - 1).mean(axis=0)
diff_p99 = np.quantile(abs_diff.reshape(-1), 0.99)

plt.figure(figsize=(10, 3))
plt.plot(diff_mean)
plt.title(f"Task1: mean(|ΔVIL|) per step (uint8), global p99≈{diff_p99:.1f}")
plt.xlabel("step t→t+1")
plt.tight_layout()
plt.show()

# Spatial intensity proxy: thresholded area fraction
thresholds = [20, 40, 80, 120]
plt.figure(figsize=(10, 3))
for thr in thresholds:
    frac = (vil_f > thr).reshape(-1, T).mean(axis=0)
    plt.plot(frac, label=f"VIL>{thr}")
plt.title("Task1: thresholded area fraction per frame")
plt.xlabel("frame t")
plt.ylabel("fraction of pixels")
plt.legend(ncol=4)
plt.tight_layout()
plt.show()


#### Core Insights
*Data Preprocessing*
- Normalization via /255 works; mean-variance standardization is also an option;
- Keep VIL’s long-tailed distribution (high values correspond to storm cores) without filtering;
- Augment high-VIL regions (e.g., cropping subgraphs) to mitigate distribution imbalance.

*Model Training*
- The model’s MAE must be lower than the baseline (0.069);
- Choose spatiotemporal modeling models like ConvLSTM/PredRNN;
- Use L1 loss to align with the evaluation metric;
- Introduce spatial weighted loss to improve prediction accuracy for high-VIL regions.

### Task 2: (vis, ir069, ir107) → vil

**EDA focus → preprocessing decision**
- **Resolution mismatch**: `vis/vil` are 384×384, but `ir069/ir107` are 192×192.
- **Fixed pipeline choice**: upsample `ir069/ir107` → **384×384** (bilinear, `align_corners=False`) so all modalities align.
- **Sanity checks**: compare pixel distribution + simple edge/gradient strength before vs after upsampling to catch strong smoothing/artifacts.

**Normalization (pipeline)**
- `vis/ir069/ir107`: `int16 -> float32`, standardize with **train-split-only** mean/std (see Section 5).
- `vil`: `uint8 -> float32`, scale by `/255.0`.

- Confirm IR resolution mismatch (192→384)
- Visualise aligned modalities at the same timestep


In [ ]:
sid = str(rng.choice(df["id"].unique()))
print("Task2 example storm:", sid)

with data.open_h5(TRAIN_H5) as f:
    vis = f[sid]["vis"][:]
    ir069 = f[sid]["ir069"][:]
    ir107 = f[sid]["ir107"][:]
    vil = f[sid]["vil"][:]

print("vis:", vis.shape, vis.dtype)
print("ir069:", ir069.shape, ir069.dtype)
print("ir107:", ir107.shape, ir107.dtype)
print("vil:", vil.shape, vil.dtype)

ti = int(rng.integers(0, 36))
fig, axs = plt.subplots(1, 4, figsize=(14, 3))
axs[0].imshow(vis[:, :, ti], cmap="gray")
axs[0].set_title("vis")
axs[1].imshow(ir069[:, :, ti], cmap="viridis")
axs[1].set_title("ir069 (192×192)")
axs[2].imshow(ir107[:, :, ti], cmap="inferno")
axs[2].set_title("ir107 (192×192)")
axs[3].imshow(vil[:, :, ti], cmap="turbo", vmin=0, vmax=255)
axs[3].set_title("vil")
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

# --- Upsampling sanity check (192→384, bilinear) ---
import torch

ir069_t = preprocess.int16_image_to_tchw(ir069[:, :, ti : ti + 1])  # (1,1,192,192)
ir107_t = preprocess.int16_image_to_tchw(ir107[:, :, ti : ti + 1])

ir069_up = preprocess.upsample_192_to_384(ir069_t).squeeze().cpu().numpy()  # (384,384)
ir107_up = preprocess.upsample_192_to_384(ir107_t).squeeze().cpu().numpy()

fig, axs = plt.subplots(2, 2, figsize=(8, 8))
axs[0, 0].imshow(ir069[:, :, ti], cmap="viridis")
axs[0, 0].set_title("ir069 original (192)")
axs[0, 1].imshow(ir069_up, cmap="viridis")
axs[0, 1].set_title("ir069 upsampled (384)")
axs[1, 0].imshow(ir107[:, :, ti], cmap="inferno")
axs[1, 0].set_title("ir107 original (192)")
axs[1, 1].imshow(ir107_up, cmap="inferno")
axs[1, 1].set_title("ir107 upsampled (384)")
for ax in axs.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

def edge_strength(a: np.ndarray) -> float:
    a = a.astype(np.float32, copy=False)
    gx = np.abs(np.diff(a, axis=1))
    gy = np.abs(np.diff(a, axis=0))
    return float(gx.mean() + gy.mean())

print("ir069 edge strength: orig=", edge_strength(ir069[:, :, ti]), " up=", edge_strength(ir069_up))
print("ir107 edge strength: orig=", edge_strength(ir107[:, :, ti]), " up=", edge_strength(ir107_up))

fig, axs = plt.subplots(2, 2, figsize=(10, 6))
axs[0, 0].hist(ir069[:, :, ti].ravel(), bins=80)
axs[0, 0].set_title("ir069 original distribution")
axs[0, 1].hist(ir069_up.ravel(), bins=80)
axs[0, 1].set_title("ir069 upsampled distribution")
axs[1, 0].hist(ir107[:, :, ti].ravel(), bins=80)
axs[1, 0].set_title("ir107 original distribution")
axs[1, 1].hist(ir107_up.ravel(), bins=80)
axs[1, 1].set_title("ir107 upsampled distribution")
plt.tight_layout()
plt.show()


#### Core Insights
**Data Preprocessing**
- Must upsample ir069/ir107 to 384×384, unifying resolution with vis/vil;
- Bilinear interpolation is a feasible upsampling method (good distribution preservation);
- Normalize different modalities separately (vis/ir are int16, vil is uint8—large value range differences).

**Model Training**
- The model must support multi-modal feature fusion (after resolution unification);
- Strengthen cross-modal information complementarity (e.g., atmospheric features of ir assist VIL prediction);
- Add a modal spatial alignment module to improve multi-modal feature matching.


### Task 3: Classification (event_type)

**EDA focus → preprocessing/training decision**
- **Class imbalance**: check storm-level `event_type` distribution (Section 3.1). If long-tail exists, prefer **Macro-F1** and consider class-weighting or balanced sampling.
- **Interpretable feature baseline**: compute simple time/space stats from inputs (e.g. VIL mean/max, threshold area, temporal change) to sanity-check label signal.
- **Separability visualisation**: PCA/UMAP on baseline features (colored by class) to see which classes are inherently separable.

- Class imbalance and example storms per class
- Use this to justify Macro-F1 vs Accuracy and whether class weighting is needed


In [ ]:
# Show one random example storm per class (VIL frame as a quick visual)
examples = (
    storm_labels.groupby("event_type")["id"].apply(lambda s: s.sample(1, random_state=SEED)).to_dict()
)

with data.open_h5(TRAIN_H5) as f:
    fig, axs = plt.subplots(2, 4, figsize=(14, 6))
    axs = axs.ravel()
    for ax, (evt, sid) in zip(axs, examples.items()):
        vil = f[str(sid)]["vil"][:, :, 10]
        ax.imshow(vil, cmap="turbo", vmin=0, vmax=255)
        ax.set_title(evt)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

examples

# --- Interpretable feature baseline (sampled) ---
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

n_storms = 300
storm_ids = rng.choice(np.array(storm_labels["id"].astype(str)), size=min(n_storms, len(storm_labels)), replace=False)
label_map = dict(zip(storm_labels["id"].astype(str), storm_labels["event_type"].astype(str)))

rows = []
with data.open_h5(TRAIN_H5) as f:
    for sid in storm_ids:
        vil = f[str(sid)]["vil"][:].astype(np.float32)  # (384,384,36) uint8 -> float32
        T = vil.shape[2]
        frame_mean = vil.reshape(-1, T).mean(axis=0)
        # simple spatial intensity proxies
        area80 = (vil > 80).reshape(-1, T).mean(axis=0)
        # temporal change proxy
        diff_mean = np.abs(vil[:, :, 1:] - vil[:, :, :-1]).reshape(-1, T - 1).mean(axis=0)

        rows.append(
            {
                "id": str(sid),
                "y": label_map[str(sid)],
                "vil_mean": float(vil.mean()),
                "vil_max": float(vil.max()),
                "vil_p99": float(np.quantile(vil, 0.99)),
                "area80_mean": float(area80.mean()),
                "area80_max": float(area80.max()),
                "diff_mean": float(diff_mean.mean()),
                "diff_p99": float(np.quantile(np.abs(vil[:, :, 1:] - vil[:, :, :-1]), 0.99)),
                "peak_t": int(np.argmax(frame_mean)),
                "trend_0_12": float(frame_mean[0:12].mean() - frame_mean[12:24].mean()),
            }
        )

feat_df = pd.DataFrame(rows)
display(feat_df.head())

X = feat_df.drop(columns=["id", "y"]).to_numpy(dtype=np.float32)
y = feat_df["y"].to_numpy()

# PCA (separability glimpse)
Xz = StandardScaler().fit_transform(X)
X2 = PCA(n_components=2, random_state=SEED).fit_transform(Xz)

plt.figure(figsize=(7, 5))
for cls in sorted(np.unique(y)):
    m = y == cls
    plt.scatter(X2[m, 0], X2[m, 1], s=12, alpha=0.6, label=cls)
plt.title("Task3 baseline features: PCA (sampled)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# Simple baseline classifier to justify Macro-F1
X_train, X_val, y_train, y_val = train_test_split(
    Xz, y, test_size=0.2, random_state=SEED, stratify=y
)

clf = LogisticRegression(max_iter=2000, n_jobs=1)
clf.fit(X_train, y_train)

pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, pred, average="macro")
acc = float((pred == y_val).mean())
print("Baseline (logreg on simple features) Macro-F1:", float(macro_f1), " Accuracy:", acc)


#### Core Insights
**Data Preprocessing**
- Replace global statistical features with multi-modal (vis/ir+vil) spatiotemporal fine-grained features;
- Address class imbalance (e.g., oversample minority classes);
- Maintain feature standardization (already using StandardScaler, need to continue).
**Model Training**
- Use spatiotemporal modeling models (e.g., CNN+LSTM/Transformer) to extract more discriminative features;
- Take Macro-F1 as the core evaluation metric, adapting to multi-class + class imbalance scenarios;
- Fuse multi-modal information to improve class discrimination.

### Task 4: Lightning prediction

**EDA focus → target & submission pipeline**
- `lght` is an irregular event list `(N,5)` with columns: `[t_seconds, lat, lon, vil_pixel_x, vil_pixel_y]`.
- **Time alignment**: map `t_seconds` into **36 frame bins** (`t_frame = 0..35`) using a fixed binning rule (pipeline config).
- **Long-tail counts**: check per-storm counts; consider `log1p(count)` if needed.
- **Bounds checks**: `vil_pixel_x/y` should fall in `[0,383]` after aligning to 384×384.

**Training target**
- For each frame: produce
  - **count**: number of lightning events in that frame bin
  - **heatmap**: 384×384 rasterized counts (or smoothed heatmap)

**Submission reconstruction**
- From predicted per-frame heatmaps (and optional predicted counts), do **peak-picking / top‑K** per frame.
- Emit `(N,3)` rows as `[t_frame, x, y]` (or map back to continuous time if you decide to).
- Keep reconstruction parameters (binning seconds, threshold, max peaks, NMS radius) in a config file.

- Visualise lightning spatial density (x,y)
- Visualise lightning times t
- Use this to choose a proxy target representation (e.g. counts per frame bin)


In [ ]:
sid = str(rng.choice(df["id"].unique()))
print("Task4 example storm:", sid)

with data.open_h5(TRAIN_H5) as f:
    lght = f[sid]["lght"][:]  # (N,5) columns: t, lat, lon, x, y

print("lght shape:", lght.shape)
if lght.shape[0] == 0:
    print("No lightning events for this storm. Re-run cell to sample another.")
else:
    t = lght[:, 0]
    x = lght[:, 3]
    y = lght[:, 4]

    print("t stats: min/median/max:", float(np.min(t)), float(np.median(t)), float(np.max(t)))
    print("x/y range:", float(np.min(x)), float(np.max(x)), "/", float(np.min(y)), float(np.max(y)))

    oob = np.sum((x < 0) | (x > 383) | (y < 0) | (y > 383) | ~np.isfinite(x) | ~np.isfinite(y))
    print("out-of-bounds or non-finite x/y:", int(oob), "/", int(len(x)))

    plt.hist(t, bins=50)
    plt.title("Lightning time t (seconds)")
    plt.xlabel("t")
    plt.tight_layout()
    plt.show()

    # Heatmap of (x,y)
    H = np.zeros((384, 384), dtype=np.int32)
    xi = np.clip(x.astype(int), 0, 383)
    yi = np.clip(y.astype(int), 0, 383)
    for a, b in zip(xi, yi):
        H[b, a] += 1

    plt.figure(figsize=(5, 5))
    plt.imshow(H, cmap="magma")
    plt.title("Lightning spatial density (counts)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    # --- Align to 36 frames and rasterize per-frame targets ---
    FRAME_SECONDS = 300  # 5-min bins (make this a pipeline config)
    t_frame = np.clip((t / FRAME_SECONDS).astype(int), 0, 35)

    counts = np.bincount(t_frame, minlength=36)
    plt.figure(figsize=(10, 3))
    plt.bar(np.arange(36), counts)
    plt.title("Lightning counts per frame bin (t_frame=0..35)")
    plt.xlabel("t_frame")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

    # (T,H,W) rasterized counts
    T = 36
    heatmaps = np.zeros((T, 384, 384), dtype=np.int32)
    for tf, a, b in zip(t_frame, xi, yi):
        heatmaps[int(tf), int(b), int(a)] += 1

    top_frames = np.argsort(-counts)[:3]
    fig, axs = plt.subplots(1, len(top_frames), figsize=(12, 4))
    for ax, tf in zip(axs, top_frames):
        ax.imshow(heatmaps[int(tf)], cmap="magma")
        ax.set_title(f"heatmap t_frame={int(tf)} (count={int(counts[int(tf)])})")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    # --- Demonstrate a simple (N,3) reconstruction from heatmaps ---
    def topk_points_from_heatmap(h2: np.ndarray, k: int):
        flat = h2.reshape(-1)
        nz = np.flatnonzero(flat)
        if nz.size == 0 or k <= 0:
            return np.empty((0, 2), dtype=np.int32)
        k = int(min(k, nz.size))
        # pick top-k among non-zeros
        nz_vals = flat[nz]
        sel = nz[np.argpartition(-nz_vals, kth=k - 1)[:k]]
        ys, xs = np.divmod(sel, h2.shape[1])
        return np.stack([xs, ys], axis=1).astype(np.int32)

    rec = []
    for tf in range(T):
        k = int(counts[tf])
        pts = topk_points_from_heatmap(heatmaps[tf], k)
        for x_i, y_i in pts:
            rec.append((tf, int(x_i), int(y_i)))

    rec = np.asarray(rec, dtype=np.int32)
    print("reconstructed rows (using target heatmaps):", rec.shape)
    if rec.size:
        display(pd.DataFrame(rec, columns=["t_frame", "x", "y"]).head(10))


#### Core Insights
**Data Preprocessing**
- Fix the time alignment rule ("1 frame per 300 seconds") to map lightning time to 36 frames;
- Convert lightning point sequences into (T,H,W) heatmaps as the model's output target;
- Crop lightning coordinates to the 384×384 range to ensure spatial alignment.

**Model Training**
- Design the model to output frame-level lightning heatmaps, then extract point sequences via TopK;
- Use loss functions adapted to heatmaps (e.g., MSE), combined with L1 error of point sequences;
- Focus on optimizing prediction accuracy for frames with high lightning density.

## 5) Train/Val split and “train-only stats”

If you plan to standardize inputs (mean/std) or clip by quantiles, you should compute those **using train split only**.

Below is a helper cell that:
- builds a storm-wise stratified split
- computes simple *per-channel* mean/std estimates on a small sample (for speed)

You can increase sample sizes once it works.


In [ ]:
# Build split (storm-wise, stratified by event_type)
sp = split.make_stormwise_stratified_split(events_csv=EVENTS_CSV, val_fraction=0.2, seed=SEED, version="v1")
print("train storms:", len(sp.train_ids), "val storms:", len(sp.val_ids))

# Estimate mean/std for vis/ir channels on a small subset of TRAIN storms.
# Increase n_storms/n_frames once it's working.
n_storms = 30
n_frames = 6

storm_ids = rng.choice(np.array(sp.train_ids), size=min(n_storms, len(sp.train_ids)), replace=False)

vis_vals = []
ir069_vals = []
ir107_vals = []

with data.open_h5(TRAIN_H5) as f:
    for sid in storm_ids:
        sid = str(sid)
        vis = f[sid]["vis"][:]  # (384,384,36) int16
        ir069 = f[sid]["ir069"][:]  # (192,192,36) int16
        ir107 = f[sid]["ir107"][:]  # (192,192,36) int16

        tis = rng.choice(np.arange(vis.shape[2]), size=n_frames, replace=False)
        for ti in tis:
            vis_vals.append(vis[:, :, ti].astype(np.float32).ravel())
            ir069_vals.append(ir069[:, :, ti].astype(np.float32).ravel())
            ir107_vals.append(ir107[:, :, ti].astype(np.float32).ravel())

vis_vals = np.concatenate(vis_vals)
ir069_vals = np.concatenate(ir069_vals)
ir107_vals = np.concatenate(ir107_vals)

stats = {
    "vis": {"mean": float(vis_vals.mean()), "std": float(vis_vals.std() + 1e-6)},
    "ir069": {"mean": float(ir069_vals.mean()), "std": float(ir069_vals.std() + 1e-6)},
    "ir107": {"mean": float(ir107_vals.mean()), "std": float(ir107_vals.std() + 1e-6)},
}

import json

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

(ARTIFACTS_DIR / "train_ids.txt").write_text("\n".join(map(str, sp.train_ids)) + "\n")
(ARTIFACTS_DIR / "val_ids.txt").write_text("\n".join(map(str, sp.val_ids)) + "\n")

with (ARTIFACTS_DIR / "stats.json").open("w") as f:
    json.dump(stats, f, indent=2)

# Task4 reconstruction/config knobs (keep in sync with pipeline)
TASK4_RECONSTRUCT = {
    "frame_seconds": 300,
    "t_frame_range": [0, 35],
    "peak_method": "topk",
    "threshold": 1,
    "max_peaks_per_frame": 50,
}
with (ARTIFACTS_DIR / "task4_reconstruct.json").open("w") as f:
    json.dump(TASK4_RECONSTRUCT, f, indent=2)

print("Wrote artifacts to:", ARTIFACTS_DIR.resolve())

stats


## 6) Summary / Decisions

Use this section to write down your final choices so that training + surprise inference use the **same preprocessing**.

### Proposed preprocessing
- **VIL**: `uint8 -> float32`, scaling: `vil/255.0` (default). And quantile clipping if long-tail causes instability.
- **VIS / IR069 / IR107**: `int16 -> float32`, standardization method: z-score using **train-only** mean/std.
  - Train-only stats (mean/std or quantiles): paste values here (or read from `artifacts/stats.json`).
- **IR upsampling**: fixed `192→384` bilinear (in `preprocess.upsample_192_to_384`).
- **Task 4 target**:
  - Map lightning `t_seconds → t_frame=0..35` using `frame_seconds`.
  - Rasterize per-frame targets: `count[t]` + `heatmap[t,384,384]`.
  - Reconstruct submission `(N,3)` via per-frame peak-picking/top‑K.

### Pipeline fixed artifacts
- `artifacts/train_ids.txt`, `artifacts/val_ids.txt` (storm-wise split, stratified by `event_type`).
- `artifacts/stats.json` (train-only mean/std for `vis/ir069/ir107`).
- `artifacts/task4_reconstruct.json` (binning + reconstruction hyperparameters).

### Notes for surprise storms
- The official checker only checks `.npy` format.
- Your model quality depends on applying the **same input preprocessing** as training.
